# Frequency — OLMo-2

- OLMo-2-0425-1B
- OLMo-2-1124-7B
- OLMo-2-1124-13B

Corpus frequency (Infini-gram) for the accessibility compound battery. No model
is loaded — this is corpus-level analysis and needs no GPU.

This notebook produces **one artifact**: the per-suite frequency table
`results/frequency/olmo/olmo_frequency_table.csv`, stamped with the
corpus index it was queried against. The correlation itself is computed by
`src/dual_spearman.py`, which has the only implementation of the pre-registered
estimator (DECISIONS 2026-07-03).

## Setup

In [2]:
# Cell 1: Imports
import numpy as np
import pandas as pd
import yaml

from src.frequency import (
    COMPOUNDS, COMPOUND_DOMAINS, FAILED_COUNT,
    query_infinigram, save_suite_frequency_table,
)

## Corpus index

The index is named here as a **literal**, not looked up from a suite→index
mapping. Every path below — the query, the derived quantities, the saved table —
uses this one variable, so a robustness path cannot silently use a different
corpus.

That is not hypothetical: audit finding A4 arose from a `SUITE_INDEX` dict that
the primary path consulted correctly and the robustness paths did not.

**olmo → `v4_olmo-mix-1124_llama`** — OLMo-Mix-1124 (Llama-2 tokenizer)
Corpus match: exact — OLMo-Mix-1124 is OLMo-2's training corpus.

In [3]:
# Cell 2: Suite and corpus index — the two facts everything below depends on
SUITE = "olmo"
CORPUS_INDEX = "v4_olmo-mix-1124_llama"

print(f"suite        : {SUITE}")
print(f"corpus_index : {CORPUS_INDEX}")

suite        : olmo
corpus_index : v4_olmo-mix-1124_llama


## Scope — which compounds

The compound set is **derived from the elicitation battery**, not hardcoded and
not taken wholesale from `src.frequency.COMPOUNDS`.

`COMPOUNDS` currently holds 53 entries, but four of them (`empty_link`,
`form_label`, `link_text`, `page_title`) have no *declarative* prompt, so they
have no accuracy to correlate against. The frequency set is the battery's
declarative concepts minus the two single-token concepts (ARIA, WCAG), which
have no bigram: **51 − 2 = 49**.

Deriving it means that if the battery grows, the count printed below changes
and the assertion fires — rather than the scope moving underneath correct code,
which is how every other count in this repo has drifted.

In [4]:
# Cell 3: Derive the compound set from the battery (scope made visible)
battery = yaml.safe_load(open(PROJECT_ROOT / "data" / "accessibility.yaml"))
declarative = sorted({p["concept"] for p in battery["prompts"]
                      if p.get("prompt_type") == "declarative"})

# Concept (space form) -> compound (snake_case). `captions` is authored under
# its two-word form; see the S5 alias in DECISIONS 2026-07-03.
CONCEPT_ALIASES = {"captions": "closed_captions", "closed captions": "closed_captions"}
to_compound = lambda c: CONCEPT_ALIASES.get(c.strip().lower(),
                                            c.strip().lower().replace(" ", "_"))

# Single-token concepts have no bigram and are excluded by construction.
SINGLE_TOKEN = {"aria", "wcag"}

components = {name: (w1, w2) for name, w1, w2, _prompt in COMPOUNDS}
compounds, excluded, unmapped = [], [], []
for concept in declarative:
    comp = to_compound(concept)
    if comp in SINGLE_TOKEN:
        excluded.append(concept)
    elif comp in components:
        w1, w2 = components[comp]
        compounds.append((comp, w1, w2))
    else:
        unmapped.append(concept)

print(f"declarative concepts in battery : {len(declarative)}")
print(f"excluded (single token)         : {len(excluded)}  {excluded}")
print(f"unmapped (no entry in COMPOUNDS): {len(unmapped)}  {unmapped}")
print(f"compounds to query              : {len(compounds)}")

assert not unmapped, f"concepts with no COMPOUNDS entry: {unmapped}"
assert len(compounds) == 49, (
    f"expected 49 compounds, got {len(compounds)}. The battery changed. This is "
    f"a scope change, not a bug to silence — n moves and every Section III "
    f"number moves with it.")

declarative concepts in battery : 51
excluded (single token)         : 2  ['ARIA', 'WCAG']
unmapped (no entry in COMPOUNDS): 0  []
compounds to query              : 49


## Counts

`REQUERY = False` reuses counts that are already frozen; `True` re-queries
Infini-gram. **The default is `False` on purpose.**

Re-querying changes `x` and therefore moves every number Section III reports.
That is a decision with its own ADR, not a side effect of running a notebook.
The query mechanism is written out below either way, so it is readable without
being run.

**Reuse source:** `results/frequency/olmo/OLMo-2-0425-1B_spearman_merged.csv` — OLMo-Mix counts produced by the Colab run in commit `ebb4d2b`. Verified 0/49 of these match the Pile table and the count columns are identical across all three OLMo merged files, so any one of them is the same corpus table.

In [5]:
# Cell 4: Counts — query Infini-gram, or reuse frozen counts
REQUERY = False

screened = []   # (compound, field) whose lookup failed; kept, not swallowed

def count_of(response, compound, field):
    """Extract a count, screening the failure sentinel to NaN.

    Infini-gram returns FAILED_COUNT (-1) when a lookup fails after all
    retries. It is not caught by dropna(), and as the smallest value it would
    take the bottom rank in a Spearman — i.e. a failed lookup would read as
    'rarest compound'. Screen it here, before anything statistical sees it.
    """
    c = response.get("count")
    if c is None or c < 0:
        screened.append((compound, field))
        return np.nan
    return float(c)

if REQUERY:
    rows = []
    for i, (comp, w1, w2) in enumerate(compounds, 1):
        print(f"[{i:>2}/{len(compounds)}] {comp}")
        bg = query_infinigram(f"{w1} {w2}", index=CORPUS_INDEX)
        u1 = query_infinigram(w1, index=CORPUS_INDEX)
        u2 = query_infinigram(w2, index=CORPUS_INDEX)
        rows.append({
            "compound": comp, "word1": w1, "word2": w2,
            "bigram_count": count_of(bg, comp, "bigram_count"),
            "word1_count":  count_of(u1, comp, "word1_count"),
            "word2_count":  count_of(u2, comp, "word2_count"),
        })
    freq = pd.DataFrame(rows)
else:
    merged = pd.read_csv(PROJECT_ROOT / "results" / "frequency" / "olmo" /
                         "OLMo-2-0425-1B_spearman_merged.csv")
    freq = merged[["compound", "word1", "word2",
                   "bigram_count", "word1_count", "word2_count"]].copy()
    print(f"reused OLMo-Mix counts: {len(freq)} rows")

# Restrict to the derived scope and attach domain labels.
wanted = [c for c, _, _ in compounds]
freq = freq[freq["compound"].isin(wanted)].copy()
freq["domain"] = freq["compound"].map(COMPOUND_DOMAINS)

print(f"\ncompounds queried : {len(compounds)}")
print(f"compounds returned: {len(freq)}")
print(f"rows screened as {FAILED_COUNT}: {len(screened)}  {screened if screened else ''}")
assert len(freq) == len(compounds), (
    f"{len(compounds) - len(freq)} compound(s) missing from the count source")

reused OLMo-Mix counts: 49 rows

compounds queried : 49
compounds returned: 49
rows screened as -1: 0  


## Derived quantities

Two quantities are computed from the raw counts.

**Conditional probability** `P(word2 | word1) = count(bigram) / count(word1)`.
The guard is `is not None`, **not** truthiness. A conditional probability of
exactly `0.0` is a real measurement — the two words never co-occur — and
`if cond_prob:` would record that as *missing*, which is a different claim. That
bug is audit finding A18.

**PMI** `log2( P(w1,w2) / (P(w1)·P(w2)) )`. Corpus size cancels out of the
ranking, so the rank-invariant kernel `log2( bigram / (w1·w2) )` is used and
labelled as such — matching `src/closeout_followups.pmi_robustness`.

In [6]:
# Cell 5: Conditional probability and PMI
def conditional_probability(bigram_count, word1_count):
    """P(word2 | word1). Returns None when undefined, 0.0 when genuinely zero."""
    if pd.isna(bigram_count) or pd.isna(word1_count) or word1_count <= 0:
        return None
    return bigram_count / word1_count

cond = [conditional_probability(b, w)
        for b, w in zip(freq["bigram_count"], freq["word1_count"])]

# `is not None` — a legitimate 0.0 must survive as 0.0, not become missing.
freq["conditional_prob"] = [round(c, 6) if c is not None else np.nan for c in cond]

# Rank-invariant PMI kernel; NaN counts propagate rather than being imputed.
freq["pmi"] = np.log2(freq["bigram_count"] /
                      (freq["word1_count"] * freq["word2_count"]))

n_zero = int((freq["conditional_prob"] == 0.0).sum())
print(f"conditional_prob: {freq['conditional_prob'].notna().sum()} defined, "
      f"{n_zero} of them exactly 0.0 (retained, not treated as missing)")
print(f"pmi             : {freq['pmi'].notna().sum()} defined")
freq[["compound", "bigram_count", "word1_count", "conditional_prob", "pmi"]].head()

conditional_prob: 49 defined, 0 of them exactly 0.0 (retained, not treated as missing)
pmi             : 49 defined


,compound,bigram_count,word1_count,conditional_prob,pmi
0,accessibility_tree,27026,17244920,0.001567,-37.100587
1,accessible_authentication,479,73358835,0.000007,-41.935651
2,accessible_description,13071,73358835,0.000178,-39.583870
3,accessible_name,45152,73358835,0.000615,-40.590323
4,alt_text,1510749,72461750,0.020849,-34.416984


## Write the per-suite table

One suite, one file, one writer. `save_suite_frequency_table` stamps every row
with `corpus_index`, so the corpus that produced a number is recoverable from
the artifact itself rather than from the notebook that produced it.

It raises if a negative count survived — i.e. if the sentinel screen above was
skipped — rather than writing a `-1` that a later Spearman would rank.

Nothing here writes the global `results/frequency/frequency_table.csv`. That
file is a frozen Pile artifact; a pipeline that can overwrite it makes it not
frozen.

In [7]:
# Cell 6: Write results/frequency/{SUITE}/{SUITE}_frequency_table.csv
path = save_suite_frequency_table(freq, PROJECT_ROOT, SUITE, CORPUS_INDEX)
pd.read_csv(path).head()

olmo: wrote 49 rows -> /Users/trishasalas/Repos/Research/tmlr/results/frequency/olmo/olmo_frequency_table.csv
olmo: corpus_index = v4_olmo-mix-1124_llama
olmo: 0 row(s) have no usable bigram count


,compound,domain,word1,word2,bigram_count,word1_count,word2_count,conditional_prob,pmi,corpus_index
0,accessibility_tree,accessibility,accessibility,tree,27026,17244920,230945885,0.001567,-37.100587,v4_olmo-mix-1124_llama
1,accessible_authentication,accessibility,accessible,authentication,479,73358835,27464520,0.000007,-41.935651,v4_olmo-mix-1124_llama
2,accessible_description,accessibility,accessible,description,13071,73358835,146821187,0.000178,-39.583870,v4_olmo-mix-1124_llama
3,accessible_name,accessibility,accessible,name,45152,73358835,1018894452,0.000615,-40.590323,v4_olmo-mix-1124_llama
4,alt_text,accessibility,alt,text,1510749,72461750,478220452,0.020849,-34.416984,v4_olmo-mix-1124_llama


## Correlation

The estimator is **not** implemented here. `src/dual_spearman.py` holds the only
implementation of the pre-registered spec (DECISIONS 2026-07-03): one
observation per compound, `x = log10(bigram_count)`, `y = ` mean strict binary
accuracy across scales, with Kendall / partial / secondary / bootstrap
robustness alongside.

Three notebooks each computing their own Spearman is how the number carrying
Section III ends up in three places that can drift — which is exactly what
happened: a second estimator in `src/frequency.py` wrote its result to
`spearman_summary.csv`, the same path `dual_spearman` writes as PRIMARY.

`run_suite` reads the per-suite table written above, so this correlation uses
`v4_olmo-mix-1124_llama` by construction.

In [8]:
# Cell 7: Run the pre-registered estimator for this suite
from src.dual_spearman import run_suite

result = run_suite(PROJECT_ROOT, SUITE)

print(f"\ncorpus_index used: {result['corpus_index']}")
print("\nPRIMARY")
print(result["primary"].to_string(index=False))
print("\nKendall")
print(result["kendall"].to_string(index=False))
print("\nPartial")
print(result["partial"].to_string(index=False))

Loaded:
  Elicitation: 3458 rows across 13 models (['gpt2', 'olmo', 'pythia'])  source={'original': 2925, 'expansion': 533}
  Entropy: 3284 rows across 13 models (['gpt2', 'olmo', 'pythia'])  source={'original': 2751, 'expansion': 533}
  Binding: 2150144 rows across 13 models (['gpt2', 'olmo', 'pythia'])  source={'original': 1723904, 'expansion': 426240}

Skipped 13 CSV(s) — stem did not resolve to a KNOWN_DOMAIN:
  results/entropy/gpt2/gpt2-large/gpt2-large-entropy.csv  ->  domain='entropy'
  results/entropy/gpt2/gpt2-medium/gpt2-medium-entropy.csv  ->  domain='entropy'
  results/entropy/gpt2/gpt2-small/gpt2-entropy.csv  ->  domain='gpt2-entropy'
  results/entropy/gpt2/gpt2-xl/gpt2-xl-entropy.csv  ->  domain='entropy'
  results/entropy/olmo/OLMo-2-0425-1B/OLMo-2-0425-1B-entropy.csv  ->  domain='entropy'
  results/entropy/olmo/OLMo-2-1124-13B/OLMo-2-1124-13B-entropy.csv  ->  domain='entropy'
  results/entropy/olmo/OLMo-2-1124-7B/OLMo-2-1124-7B-entropy.csv  ->  domain='entropy'
  result